# Solution 09 — a read-only recipe server

The agent can list and read recipes, never sees `delete_recipe`, and recovers from a wrong name
through the server's `ToolError`. Every claim is a plain `assert`, and nothing needs a key.

In [ ]:
from learningdantic import setup

MODEL = setup()

In [ ]:
from mcp.server.mcpserver import MCPServer
from mcp.server.mcpserver.exceptions import ToolError
from pydantic_ai import Agent
from pydantic_ai.mcp import MCPToolset
from pydantic_ai.models.test import TestModel


def offered(test_model):
    """Names of the tools the model was shown on its last request."""
    return sorted(tool.name for tool in test_model.last_model_request_parameters.function_tools)


RECIPES = {
    "pancakes": "Whisk 1 cup flour, 1 egg and 1 cup milk. Fry spoonfuls in butter.",
    "omelette": "Beat 3 eggs, pour into a hot buttered pan, fold when set.",
}
recipes = MCPServer(name="recipes", log_level="WARNING")


@recipes.tool()
def list_recipes() -> list[str]:
    """Names of every recipe in the book."""
    return sorted(RECIPES)


@recipes.tool()
def get_recipe(name: str) -> str:
    """The full recipe for one name from list_recipes."""
    if name.lower() not in RECIPES:
        raise ToolError(f"No recipe called {name!r}. Known recipes: {', '.join(sorted(RECIPES))}.")
    return RECIPES[name.lower()]


@recipes.tool()
def delete_recipe(name: str) -> str:
    """Delete a recipe for good."""
    RECIPES.pop(name.lower(), None)
    return f"deleted {name}"


all_tools = MCPToolset(recipes)
read_only = all_tools.filtered(lambda ctx, td: td.name != "delete_recipe")

full_model, scoped_model = TestModel(call_tools=[]), TestModel(call_tools=[])
await Agent(full_model, toolsets=[all_tools]).run("Delete the pancakes recipe.")
await Agent(scoped_model, toolsets=[read_only]).run("Delete the pancakes recipe.")
print("server has:", offered(full_model))
print("model sees:", offered(scoped_model))

# Part 1: the server really has delete_recipe, and the filter is what hides it.
assert "delete_recipe" in offered(full_model), offered(full_model)
assert offered(scoped_model) == ["get_recipe", "list_recipes"], offered(scoped_model)

In [ ]:
from pydantic_ai.messages import (
    ModelResponse,
    RetryPromptPart,
    TextPart,
    ToolCallPart,
    ToolReturnPart,
)
from pydantic_ai.models.function import FunctionModel


def typo_then_fix(messages, info):
    """Ask for 'pancake', retry with 'pancakes' after the error, then answer with the recipe."""
    last = messages[-1].parts
    if any(isinstance(part, RetryPromptPart) for part in last):
        return ModelResponse(parts=[ToolCallPart("get_recipe", {"name": "pancakes"})])
    if any(isinstance(part, ToolReturnPart) for part in last):
        return ModelResponse(parts=[TextPart(f"Here is the recipe: {last[0].content}")])
    return ModelResponse(parts=[ToolCallPart("get_recipe", {"name": "pancake"})])


agent = Agent(FunctionModel(typo_then_fix), toolsets=[read_only])
result = await agent.run("How do I make pancakes?")
parts = [part for message in result.all_messages() for part in message.parts]
kinds = [type(part).__name__ for part in parts]
retry = next(part for part in parts if isinstance(part, RetryPromptPart))
print("flow  :", kinds)
print("retry :", retry.content)
print("answer:", result.output)

# Part 2: error -> retry -> answer, and the error told the model the right name.
expected = ["UserPromptPart", "ToolCallPart", "RetryPromptPart", "ToolCallPart", "ToolReturnPart"]
assert kinds == [*expected, "TextPart"], kinds
assert "pancakes" in str(retry.content), retry.content
assert RECIPES["pancakes"] in result.output, result.output

In [ ]:
from dataclasses import dataclass


@dataclass
class Reader:
    role: str  # "guest" or "owner"


model = TestModel(call_tools=[])
book = Agent(
    model,
    deps_type=Reader,
    toolsets=[
        all_tools.filtered(lambda ctx, td: td.name != "delete_recipe" or ctx.deps.role == "owner")
    ],
)
seen = {}
for role in ["guest", "owner"]:
    await book.run("Delete the omelette recipe.", deps=Reader(role))
    seen[role] = offered(model)
print("seen by role:", seen)

# Stretch: only the owner is shown delete_recipe, and nothing was deleted along the way.
assert seen["guest"] == ["get_recipe", "list_recipes"], seen
assert seen["owner"] == ["delete_recipe", "get_recipe", "list_recipes"], seen
assert sorted(RECIPES) == ["omelette", "pancakes"], RECIPES